In [ ]:
!pip install transformers datasets scikit-learn pandas numpy torch

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from google.colab import files

uploaded = files.upload()   # opens a file picker

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.metrics import f1_score

# ✅ Check GPU
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

# ✅ Load model & tokenizer
model_name = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4,
    problem_type="multi_label_classification"
)

# ✅ Load & clean data
df = pd.read_csv("data.csv", encoding="latin-1")
df["Abstract"] = df["Abstract"].astype(str)
label_cols = ["Patient", "Reporter", "Drug", "Event"]
df = df.dropna(subset=label_cols)
df[label_cols] = df[label_cols].astype(int)
print(f"Total rows after cleaning: {len(df)}")

# ✅ Create & split dataset
dataset = Dataset.from_pandas(df, preserve_index=False)
dataset_split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset_split["train"]
eval_dataset  = dataset_split["test"]
print(f"Train size: {len(train_dataset)} | Eval size: {len(eval_dataset)}")

# ✅ Tokenize
def tokenize(example):
    return tokenizer(
        example["Abstract"],
        truncation=True,
        padding=False,
        max_length=512
    )

train_dataset = train_dataset.map(tokenize, batched=True)
eval_dataset  = eval_dataset.map(tokenize, batched=True)

# ✅ Format labels
def format_labels(example):
    example["labels"] = [
        float(example["Patient"]),
        float(example["Reporter"]),
        float(example["Drug"]),
        float(example["Event"])
    ]
    return example

train_dataset = train_dataset.map(format_labels)
eval_dataset  = eval_dataset.map(format_labels)

# ✅ Set torch format
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# ✅ Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ✅ Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)
    return {
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0)
    }

# ✅ Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
)

# ✅ Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# ✅ Train
trainer.train()

# ✅ Save
trainer.save_model("./NeoPharmBert")
tokenizer.save_pretrained("./NeoPharmBert")
print("Model saved to ./NeoPharmBert ✅")

In [ ]:
from sklearn.metrics import classification_report, f1_score
import json
import numpy as np

# ✅ Run predictions on eval set
predictions = trainer.predict(eval_dataset)
probs = 1 / (1 + np.exp(-predictions.predictions))
preds = (probs > 0.5).astype(int)
labels = predictions.label_ids.astype(int)

# ✅ Per-label classification report
print(classification_report(
    labels, preds,
    target_names=["Patient", "Reporter", "Drug", "Event"]
))

# ✅ Overall scores
f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
f1_micro = f1_score(labels, preds, average="micro", zero_division=0)

print(f"F1 Macro: {f1_macro:.4f}")
print(f"F1 Micro: {f1_micro:.4f}")

# ✅ Save results summary
results = {
    "model": "NeoPharmabert - PubMedBERT fine-tuned for ICSR validity detection",
    "dataset_size": len(df),
    "train_size": len(train_dataset),
    "eval_size": len(eval_dataset),
    "best_epoch": 4,
    "f1_macro": round(f1_macro, 4),
    "f1_micro": round(f1_micro, 4),
    "labels": ["Patient", "Reporter", "Drug", "Event"]
}

with open("./NeoPharmBert/results_summary.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nResults saved to ./NeoPharmBert/results_summary.json ✅")



In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification


model_name = "/content/NeoPharmBert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
print("NeoPharmabert loaded ")

# Inference function
def predict_icsr(abstract_text):
    inputs = tokenizer(
        abstract_text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.sigmoid(outputs.logits).squeeze().cpu().numpy()
    preds = (probs > 0.5).astype(int)

    criteria = ["Patient", "Reporter", "Drug", "Event"]

    print("\n--- ICSR Validity Prediction ---")
    for label, prob, pred in zip(criteria, probs, preds):
        status = "Yes" if pred == 1 else "No"
        print(f"{label:<12}: {status}")

    overall = "VALID ICSR" if all(preds) else "INVALID ICSR"
    print(f"\nOverall      : {overall}")



abstract = """
1. Contemp Clin Dent. 2015 Sep;6(Suppl 1):S278-81. doi: 10.4103/0976-237X.166838.

Paracetamol induced Steven-Johnson syndrome: A rare case report.

Rajput R(1), Sagari S(2), Durgavanshi A(3), Kanwar A(4).

Author information:
(1)Department of Oral Medicine and Radiology, Jodhpur Dental College General
Hospital, Jodhpur, Rajasthan, India.
(2)Department of Oral and Maxillofacial Pathology, Jodhpur Dental College
General Hospital, Jodhpur, Rajasthan, India.
(3)Department of Oral Medicine and Radiology, IDS Dental college and Hospital,
Bareilly, Uttar Pradesh, India.
(4)Department of Oral and Maxillofacial Pathology, NIMS Dental College General
Hospital, Jaipur, Rajasthan, India.

In the contemporary era, use of drugs is the dominant paradigm of health care.
The most quotidian drug used for fever and pain is paracetamol. Although adverse
reactions to paracetamol in India are rare, at times they can cause
life-threatening situations. Stevens-Johnson syndrome (SJS) is one such
potentially lethal adverse drug reaction. The most reported cases of
analgesic-induced SJS were due to oxicams or propionic acid derivatives. There
are very few detailed reports of SJS due to the use of paracetamol. We report a
case of SJS, which occurred due to the use of paracetamol. The clinical features
of this condition and multidisciplinary management of the patient are described
in brief.

DOI: 10.4103/0976-237X.166838
PMCID: PMC4632237
PMID: 26604588
"""

predict_icsr(abstract)

In [ ]:
import shutil

# ✅ Backup to Google Drive
shutil.copytree(
    "/content/NeoPharmBert",
    "/content/drive/MyDrive/NeoPharmBert",
    dirs_exist_ok=True
)
print("✅ Backed up to Google Drive!")

In [ ]:
!pip install huggingface_hub -q

from huggingface_hub import login
login(token="")  # Paste your HF token when prompted

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_path = "/content/drive/MyDrive/NeoPharmBert"
HF_TOKEN = ""  # paste your token here
HF_REPO = "itz-dhanu/NeoPharmabert"  # replace with your HF username

# Load from Drive
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Push directly with token
model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)

print("✅ Uploaded to HuggingFace!")